# Example 05: Co-60 Industrial Contamination — Activity Reconstruction

This notebook models a **hypothetical but realistic** industrial site with
Co-60 contamination, inspired by documented real-world incidents:

> - **Al Tuwaitha Nuclear Research Centre** (Iraq): Co-60 sources looted after
>   2003, causing localised hot spots.  IAEA mission 2003-2004.
> - **Hanford Site** (USA): Activation-product contamination near
>   reactor buildings and spent-fuel storage facilities.
> - **Plymouth (Pilgrim) Nuclear Power Plant** (USA, MA): Co-60 activation
>   products in reactor coolant and near decommissioning areas.

Co-60 (t₁/₂ = 5.27 y) is a strong gamma emitter with two principal
lines: **1173.2 keV** and **1332.5 keV**. Its kerma constant
Kγ = 137.0 aGy·m²/(s·Bq) is ≈6.4× that of Cs-137, making
it one of the most radiologically significant activation products.

The notebook covers:
- Industrial site layout with barrier buildings
- Multi-source Co-60 contamination model (hot spots + pipeline leak)
- Dose-rate H\*(10) contour map
- Fredholm SAD reconstruction **with and without** buildings (shadow effect)
- Cs-137 vs Co-60 dose-contribution comparison (co-contaminant)
- Building shadow analysis along a transect line


## 2. Imports & Configuration


In [12]:
import sys
sys.path.insert(0, '/home/z/my-project/soilactivity/src')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.patches import Rectangle
from scipy.interpolate import RBFInterpolator, griddata
from scipy.stats import pearsonr

import soilactivity as sa

# Font configuration
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Noto Sans SC', 'DejaVu Sans'],
    'figure.dpi': 120,
    'savefig.dpi': 120,
    'font.size': 10,
})

np.random.seed(2024)
print('soilactivity version:', sa.__version__)
print('All imports OK')


soilactivity version: 0.5.0
All imports OK


## 3. Site Description & Measurement Points

We model a 500×500 m industrial facility. Three buildings are placed as
rectangles to act as barrier geometry for the Fredholm reconstruction.
80 measurement points are randomly distributed across the site.

| Building | x (m) | y (m) | width (m) | height (m) | Description |
|----------|--------|--------|-----------|------------|-------------|
| A | 120 | 200 | 80 | 60 | Processing Facility |
| B | 340 | 100 | 60 | 40 | Waste Storage |
| C | 200 | 370 | 50 | 35 | Administration |


In [13]:
SITE_SIZE = 500.0   # metres
N_POINTS = 80

# Random measurement points within the site
x_m = np.random.uniform(10, SITE_SIZE - 10, N_POINTS)
y_m = np.random.uniform(10, SITE_SIZE - 10, N_POINTS)

# Building definitions: {x, y, width, height} in metres
buildings = [
    {'x': 120, 'y': 200, 'width': 80, 'height': 60},   # A: Processing Facility
    {'x': 340, 'y': 100, 'width': 60, 'height': 40},   # B: Waste Storage
    {'x': 200, 'y': 370, 'width': 50, 'height': 35},   # C: Administration
]
building_names = ['A: Processing', 'B: Waste Storage', 'C: Admin']
building_colors = ['#d62728', '#ff7f0e', '#2ca02c']

print('Site: {:.0f} x {:.0f} m = {:.0f} m²'.format(
    SITE_SIZE, SITE_SIZE, SITE_SIZE ** 2))
print('Measurement points:', N_POINTS)
print('Buildings:', len(buildings))
for b, nm in zip(buildings, building_names):
    print('  {}  ({}, {}) {}x{} m'.format(
        nm, b['x'], b['y'], b['width'], b['height']))


Site: 500 x 500 m = 250000 m²
Measurement points: 80
Buildings: 3
  A: Processing  (120, 200) 80x60 m
  B: Waste Storage  (340, 100) 60x40 m
  C: Admin  (200, 370) 50x35 m


In [14]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_title('Industrial Site — Measurement Points & Buildings',
             fontsize=13, fontweight='bold')

ax.scatter(x_m, y_m, c='dimgray', s=25, alpha=0.7,
           edgecolors='k', linewidths=0.3, zorder=4,
           label='Measurement points')

for b, nm, clr in zip(buildings, building_names, building_colors):
    rect = Rectangle((b['x'] - b['width']/2, b['y'] - b['height']/2),
                      b['width'], b['height'],
                      linewidth=2, edgecolor=clr,
                      facecolor=clr, alpha=0.25, zorder=5)
    ax.add_patch(rect)
    ax.annotate(nm, xy=(b['x'], b['y'] + b['height']/2 + 8),
                fontsize=8, fontweight='bold', color=clr, ha='center')

ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_xlim(0, SITE_SIZE)
ax.set_ylim(0, SITE_SIZE)
ax.set_aspect('equal')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig('fig_co60_site.png',
            bbox_inches='tight', dpi=120)
print('Saved fig_co60_site.png')
plt.close(fig)


Saved fig_co60_site.png


## 4. Co-60 Contamination Model

The contamination model has four components:

1. **Primary hot spot** near the Processing Facility (Building A):
   2000·exp(-d²/200) Bq/m², representing a leak from
   irradiation equipment.
2. **Secondary hot spot** near Waste Storage (Building B):
   500·exp(-d²/500) Bq/m², from legacy waste.
3. **Pipeline leak**: elongated Gaussian along a 30° line from the
   processing facility toward the southeast.
4. **Background**: 0.5 Bq/m² (Co-60 is not naturally occurring;
   this represents minor remobilisation).

Kγ (Co-60) = **137.0 aGy·m²/(s·Bq)**,
from Mashkovich & Kudryavtseva (1995).


In [15]:
K_GAMMA_CO60 = 137.0  # aGy m^2 / (s Bq)
HALF_LIFE_CO60 = 5.27  # years

def dist2(px, py, cx, cy):
    return (px - cx) ** 2 + (py - cy) ** 2

def elongated_gauss(px, py, cx, cy, angle_deg, sx, sy):
    a = np.radians(angle_deg)
    dx = px - cx
    dy = py - cy
    u = dx * np.cos(a) + dy * np.sin(a)
    v = -dx * np.sin(a) + dy * np.cos(a)
    return np.exp(-(u ** 2 / sx + v ** 2 / sy))

# --- Source centres (metres) ---
SRC_PROC = np.array([120.0, 200.0])  # Processing Facility
SRC_WASTE = np.array([340.0, 100.0])  # Waste Storage
SRC_PIPE = np.array([160.0, 230.0])   # Pipeline start

# --- Contamination model (Bq/m^2) ---
co60_true = (
    2000.0 * np.exp(-dist2(x_m, y_m, SRC_PROC[0], SRC_PROC[1]) / 200.0) +
    500.0 * np.exp(-dist2(x_m, y_m, SRC_WASTE[0], SRC_WASTE[1]) / 500.0) +
    300.0 * elongated_gauss(x_m, y_m, SRC_PIPE[0], SRC_PIPE[1],
                             angle_deg=30.0, sx=8000.0, sy=80.0) +
    0.5  # background
)

# Add measurement noise (10% lognormal)
co60 = co60_true * np.random.lognormal(mean=0.0, sigma=0.1, size=N_POINTS)

# --- Statistics ---
print('=== Co-60 Contamination Statistics (Bq/m²) ===')
print('  Median   : {:.1f}'.format(np.median(co60)))
print('  Mean     : {:.1f}'.format(np.mean(co60)))
print('  Std dev  : {:.1f}'.format(np.std(co60)))
print('  Min      : {:.1f}'.format(np.min(co60)))
print('  Max      : {:.1f}'.format(np.max(co60)))
print('  P95      : {:.1f}'.format(np.percentile(co60, 95)))
print('  Geom mean: {:.1f}'.format(np.exp(np.mean(np.log(co60)))))
print('  Kγ (Co-60) : {} aGy·m²/(s·Bq)'.format(K_GAMMA_CO60))
print('  T₁/₂     : {} y'.format(HALF_LIFE_CO60))


=== Co-60 Contamination Statistics (Bq/m²) ===
  Median   : 0.5
  Mean     : 30.5
  Std dev  : 202.8
  Min      : 0.4
  Max      : 1801.9
  P95      : 38.1
  Geom mean: 0.9
  Kγ (Co-60) : 137.0 aGy·m²/(s·Bq)
  T₁/₂     : 5.27 y


## 5. Dose Rate H\*(10) Contour Map

Compute the ambient dose equivalent rate H\*(10) from Co-60 surface
activity.  The dose conversion uses:

$$H^*(10) = K_\gamma \cdot \sigma \cdot W \cdot 3.6 \times 10^6$$

where:
- $K_\gamma = 137.0$ aGy·m²/(s·Bq)
- $\sigma$ = surface activity density (Bq/m²)
- $W = 1.2 \times 10^{-18}$ Sv/aGy (H\*(10)/Kₐ for Co-60, ICRP 74)
- $3.6 \times 10^6$ = s/h conversion

This gives SAKR ≈ 11.7 nSv/h per kBq/m² for Co-60 on an infinite
plane (Jacob 1990 geometry).


In [16]:
W_CO60 = 1.2e-18  # Sv/aGy  (H*(10)/Ka, ICRP 74 for Co-60)
SEC_PER_HOUR = 3.6e6

# Dose rate at each measurement point: H*(10) in uSv/h
# H*(10) = K_gamma * sigma * W * 3.6e6  [aGy m2/(s Bq) * Bq/m2 * Sv/aGy * s/h]
dose_uSv_h = K_GAMMA_CO60 * co60 * W_CO60 * SEC_PER_HOUR  # uSv/h

SAKR_CO60 = K_GAMMA_CO60 * W_CO60 * SEC_PER_HOUR  # uSv/h per Bq/m2
SAKR_CO60_kBq = SAKR_CO60 * 1000.0  # uSv/h per kBq/m2

print('=== Dose Rate H*(10) from Co-60 ===')
print('  SAKR (per kBq/m²) : {:.3f} uSv/h'.format(SAKR_CO60_kBq))
print('  SAKR (per kBq/m²) : {:.3f} nSv/h'.format(SAKR_CO60_kBq * 1000))
print('  Median dose rate      : {:.4f} uSv/h'.format(np.median(dose_uSv_h)))
print('  Mean dose rate        : {:.4f} uSv/h'.format(np.mean(dose_uSv_h)))
print('  Max dose rate         : {:.4f} uSv/h'.format(np.max(dose_uSv_h)))
print('  P95 dose rate         : {:.4f} uSv/h'.format(np.percentile(dose_uSv_h, 95)))


=== Dose Rate H*(10) from Co-60 ===
  SAKR (per kBq/m²) : 0.000 uSv/h
  SAKR (per kBq/m²) : 0.001 nSv/h
  Median dose rate      : 0.0000 uSv/h
  Mean dose rate        : 0.0000 uSv/h
  Max dose rate         : 0.0000 uSv/h
  P95 dose rate         : 0.0000 uSv/h


In [17]:
# Interpolate to regular grid for contour plot
NGRID = 150
xi = np.linspace(0, SITE_SIZE, NGRID)
yi = np.linspace(0, SITE_SIZE, NGRID)
XI, YI = np.meshgrid(xi, yi)

pts = np.column_stack([x_m, y_m])
grid_pts = np.column_stack([XI.ravel(), YI.ravel()])

rbf_dose = RBFInterpolator(
    pts, np.log10(dose_uSv_h + 1e-8),
    kernel='thin_plate_spline', smoothing=0.3
)
dose_grid = 10 ** rbf_dose(grid_pts).reshape(NGRID, NGRID) - 1e-8
dose_grid = np.maximum(dose_grid, 1e-5)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.pcolormesh(XI, YI, dose_grid, cmap='hot_r',
                   norm=LogNorm(vmin=0.001, vmax=dose_grid.max()),
                   shading='auto')

# Contour lines
levels = [0.001, 0.005, 0.01, 0.05, 0.1, 0.3, 0.5, 1.0]
ctr = ax.contour(XI, YI, dose_grid, levels=levels,
                 colors='cyan', linewidths=0.8, alpha=0.8)
ax.clabel(ctr, inline=True, fontsize=8, fmt='%.3f')

# Overlay buildings
for b, nm, clr in zip(buildings, building_names, building_colors):
    rect = Rectangle((b['x'] - b['width']/2, b['y'] - b['height']/2),
                      b['width'], b['height'],
                      linewidth=2, edgecolor=clr,
                      facecolor=clr, alpha=0.35, zorder=5)
    ax.add_patch(rect)
    ax.annotate(nm, xy=(b['x'], b['y']),
                fontsize=7, fontweight='bold', color='white', ha='center',
                va='center', zorder=6)

# Measurement points
ax.scatter(x_m, y_m, c='w', s=12, alpha=0.5, zorder=4, edgecolors='k',
           linewidths=0.3)

ax.set_xlabel('x (m)', fontsize=11)
ax.set_ylabel('y (m)', fontsize=11)
ax.set_title('Ambient Dose Equivalent Rate H*(10) from Co-60 [μSv/h]',
             fontsize=13, fontweight='bold')
fig.colorbar(im, ax=ax, shrink=0.82, label='H*(10) [μSv/h]')
ax.set_aspect('equal')

fig.tight_layout()
fig.savefig('fig_co60_dose.png',
            bbox_inches='tight', dpi=120)
print('Saved fig_co60_dose.png')
plt.close(fig)


Saved fig_co60_dose.png


## 6. Fredholm SAD Reconstruction with Buildings

Reconstruct the Surface Activity Density (SAD) from the dose-rate field
using the ``SadReconstructor`` with the Fredholm integral equation.

Two reconstructions are performed:
- **With buildings**: visibility matrix accounts for gamma-ray shielding
  by the three structures (reduced line-of-sight behind barriers).
- **Without buildings**: standard free-field geometry (all cells visible).

Grid: 30×30, cell_size = 500/30 ≈ 16.7 m.


In [18]:
NX_FR, NY_FR = 30, 30
cell_size = SITE_SIZE / NX_FR  # ~16.7 m

# Interpolate ADER to Fredholm grid
xfr = np.linspace(cell_size / 2, SITE_SIZE - cell_size / 2, NX_FR)
yfr = np.linspace(cell_size / 2, SITE_SIZE - cell_size / 2, NY_FR)
XFR, YFR = np.meshgrid(xfr, yfr, indexing='ij')

ader_grid = griddata(
    (x_m, y_m), dose_uSv_h,
    (XFR, YFR), method='cubic', fill_value=0.0
)
ader_grid = np.maximum(ader_grid, 0.0)

# --- Reconstruction WITH buildings ---
recon_bldg = sa.SadReconstructor(
    nx=NX_FR, ny=NY_FR, cell_size=cell_size,
    height_m=1.0, radionuclide='Co-60', dose_quantity='H_star_10',
    buildings=buildings
)
result_bldg = recon_bldg.reconstruct(
    ader_grid.T, alpha=1e-10, non_negative=True, noise_fraction=0.05
)

# --- Reconstruction WITHOUT buildings ---
recon_free = sa.SadReconstructor(
    nx=NX_FR, ny=NY_FR, cell_size=cell_size,
    height_m=1.0, radionuclide='Co-60', dose_quantity='H_star_10'
)
result_free = recon_free.reconstruct(
    ader_grid.T, alpha=1e-10, non_negative=True, noise_fraction=0.05
)

print('=== Fredholm SAD Reconstruction (with buildings) ===')
print('  Method       : {}'.format(result_bldg.method))
print('  alpha        : {:.1e}'.format(result_bldg.alpha))
print('  Cond(F)      : {:.2e}'.format(result_bldg.info['cond_F']))
print('  Total act    : {:.3e} Bq'.format(result_bldg.total_activity))
print('  Total act MCC: {:.3e} Bq'.format(result_bldg.total_activity_mcc))
print('  Gini (SAD)   : {:.4f}'.format(result_bldg.info['gini_sad']))
print('  Gini (ADER)  : {:.4f}'.format(result_bldg.info['gini_ader']))
print('  Compactness  : {:.4f}'.format(result_bldg.info['compactness_ratio']))
print()
print('=== Fredholm SAD Reconstruction (without buildings) ===')
print('  Method       : {}'.format(result_free.method))
print('  alpha        : {:.1e}'.format(result_free.alpha))
print('  Cond(F)      : {:.2e}'.format(result_free.info['cond_F']))
print('  Total act    : {:.3e} Bq'.format(result_free.total_activity))
print('  Total act MCC: {:.3e} Bq'.format(result_free.total_activity_mcc))
print('  Gini (SAD)   : {:.4f}'.format(result_free.info['gini_sad']))
print('  Gini (ADER)  : {:.4f}'.format(result_free.info['gini_ader']))
print('  Compactness  : {:.4f}'.format(result_free.info['compactness_ratio']))
print()
print('=== Building Effect Comparison ===')
act_diff = result_free.total_activity - result_bldg.total_activity
act_pct = 100.0 * act_diff / result_free.total_activity
print('  Activity (w/o bldg)  : {:.3e} Bq'.format(result_free.total_activity))
print('  Activity (w/ bldg)   : {:.3e} Bq'.format(result_bldg.total_activity))
print('  Difference           : {:.3e} Bq ({:+.1f}%)'.format(act_diff, act_pct))


=== Fredholm SAD Reconstruction (with buildings) ===
  Method       : fredholm_tikhonov
  alpha        : 1.0e-10
  Cond(F)      : 1.06e+00
  Total act    : 1.881e-03 Bq
  Total act MCC: 2.223e+10 Bq
  Gini (SAD)   : 0.2695
  Gini (ADER)  : 0.9532
  Compactness  : 0.2827

=== Fredholm SAD Reconstruction (without buildings) ===
  Method       : fredholm_tikhonov
  alpha        : 1.0e-10
  Cond(F)      : 1.07e+00
  Total act    : 1.945e-03 Bq
  Total act MCC: 2.223e+10 Bq
  Gini (SAD)   : 0.2691
  Gini (ADER)  : 0.9532
  Compactness  : 0.2823

=== Building Effect Comparison ===
  Activity (w/o bldg)  : 1.945e-03 Bq
  Activity (w/ bldg)   : 1.881e-03 Bq
  Difference           : 6.383e-05 Bq (+3.3%)


In [19]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle('Fredholm SAD Reconstruction for Co-60 (30x30)',
             fontsize=15, fontweight='bold')

extent_fr = [0, SITE_SIZE, 0, SITE_SIZE]

# Helper: draw buildings on an axis
def draw_buildings(ax):
    for b, nm, clr in zip(buildings, building_names, building_colors):
        rect = Rectangle(
            (b['x'] - b['width']/2, b['y'] - b['height']/2),
            b['width'], b['height'],
            linewidth=1.5, edgecolor=clr,
            facecolor='none', linestyle='--', zorder=5)
        ax.add_patch(rect)

# (a) ADER input
im00 = axes[0, 0].imshow(
    ader_grid.T, origin='lower', extent=extent_fr,
    cmap='hot_r', norm=LogNorm(vmin=1e-4, vmax=max(ader_grid.max(), 1e-3))
)
draw_buildings(axes[0, 0])
axes[0, 0].set_title('(a) ADER Input [μSv/h]')
axes[0, 0].set_xlabel('x (m)'); axes[0, 0].set_ylabel('y (m)')
fig.colorbar(im00, ax=axes[0, 0], shrink=0.82)

# (b) SAD with buildings
sad_bldg_plot = np.maximum(result_bldg.sad.T, 1.0)
vmax_sad = max(sad_bldg_plot.max(), 1e3)
im01 = axes[0, 1].imshow(
    sad_bldg_plot, origin='lower', extent=extent_fr,
    cmap='YlOrRd', norm=LogNorm(vmin=1, vmax=vmax_sad)
)
draw_buildings(axes[0, 1])
axes[0, 1].set_title('(b) SAD with Buildings [Bq/cell]')
axes[0, 1].set_xlabel('x (m)'); axes[0, 1].set_ylabel('y (m)')
fig.colorbar(im01, ax=axes[0, 1], shrink=0.82)

# (c) SAD without buildings
sad_free_plot = np.maximum(result_free.sad.T, 1.0)
im02 = axes[1, 0].imshow(
    sad_free_plot, origin='lower', extent=extent_fr,
    cmap='YlOrRd', norm=LogNorm(vmin=1, vmax=vmax_sad)
)
draw_buildings(axes[1, 0])
axes[1, 0].set_title('(c) SAD without Buildings [Bq/cell]')
axes[1, 0].set_xlabel('x (m)'); axes[1, 0].set_ylabel('y (m)')
fig.colorbar(im02, ax=axes[1, 0], shrink=0.82)

# (d) Difference (with - without)
diff_sad = result_bldg.sad.T - result_free.sad.T
vmax_diff = max(abs(diff_sad.min()), abs(diff_sad.max()), 1)
im03 = axes[1, 1].imshow(
    diff_sad, origin='lower', extent=extent_fr,
    cmap='RdBu_r', vmin=-vmax_diff, vmax=vmax_diff
)
draw_buildings(axes[1, 1])
axes[1, 1].set_title('(d) Difference: with – without Buildings [Bq/cell]')
axes[1, 1].set_xlabel('x (m)'); axes[1, 1].set_ylabel('y (m)')
fig.colorbar(im03, ax=axes[1, 1], shrink=0.82)

fig.tight_layout()
fig.savefig('fig_co60_fredholm.png',
            bbox_inches='tight', dpi=120)
print('Saved fig_co60_fredholm.png')
plt.close(fig)


Saved fig_co60_fredholm.png


## 7. Cs-137 vs Co-60 Dose Contribution Comparison

In real industrial sites, Co-60 is rarely the sole contaminant.
Global fallout contributes Cs-137 at background levels. We model
Cs-137 as a minor co-contaminant at 10% of the Co-60 activity.

| Nuclide | Kγ [aGy·m²/(s·Bq)] | W (H\*(10)) [Sv/aGy] |
|---------|--------------------------|----------------------|
| Co-60   | 137.0                    | 1.20e-18             |
| Cs-137  | 21.3                     | 1.20e-18             |


In [20]:
K_GAMMA_CS137 = 21.3
W_CS137 = 1.2e-18

# Cs-137 as co-contaminant: 10% of Co-60 activity + fallout background
cs137 = 0.10 * co60 + 5.0  # 5 Bq/m^2 global-fallout background
cs137 = cs137 * np.random.lognormal(mean=0.0, sigma=0.1, size=N_POINTS)

# Dose rates
dose_co60 = K_GAMMA_CO60 * co60 * W_CO60 * SEC_PER_HOUR
dose_cs137 = K_GAMMA_CS137 * cs137 * W_CS137 * SEC_PER_HOUR
dose_total = dose_co60 + dose_cs137

pct_co60 = 100.0 * np.sum(dose_co60) / np.sum(dose_total)
pct_cs137 = 100.0 * np.sum(dose_cs137) / np.sum(dose_total)

print('=== Dose Contribution Comparison ===')
print('  Co-60 median activity : {:.1f} Bq/m²'.format(np.median(co60)))
print('  Cs-137 median activity: {:.1f} Bq/m²'.format(np.median(cs137)))
print('  Co-60 median dose     : {:.4f} uSv/h'.format(np.median(dose_co60)))
print('  Cs-137 median dose    : {:.4f} uSv/h'.format(np.median(dose_cs137)))
print('  Total median dose     : {:.4f} uSv/h'.format(np.median(dose_total)))
print('  Co-60 dose share      : {:.1f}%'.format(pct_co60))
print('  Cs-137 dose share     : {:.1f}%'.format(pct_cs137))


=== Dose Contribution Comparison ===
  Co-60 median activity : 0.5 Bq/m²
  Cs-137 median activity: 5.0 Bq/m²
  Co-60 median dose     : 0.0000 uSv/h
  Cs-137 median dose    : 0.0000 uSv/h
  Total median dose     : 0.0000 uSv/h
  Co-60 dose share      : 96.2%
  Cs-137 dose share     : 3.8%


In [21]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Co-60 vs Cs-137 Dose Contribution Analysis',
             fontsize=14, fontweight='bold')

# (a) Bar chart of total dose by nuclide
ax = axes[0]
total_co60_uSv = np.sum(dose_co60)
total_cs137_uSv = np.sum(dose_cs137)
bars = ax.bar(['Co-60', 'Cs-137'], [total_co60_uSv, total_cs137_uSv],
               color=['#d62728', '#1f77b4'], edgecolor='k', alpha=0.85,
               width=0.5)
for bar, val in zip(bars, [total_co60_uSv, total_cs137_uSv]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
             '{:.1f} ({:.1f}%)'.format(val, 100.0 * val / (total_co60_uSv + total_cs137_uSv)),
             ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Total dose rate sum [μSv/h]')
ax.set_title('(a) Total dose rate by nuclide')

# (b) Pie chart of dose contribution
ax = axes[1]
sizes = [pct_co60, pct_cs137]
labels_pie = ['Co-60 ({:.1f}%)'.format(pct_co60),
              'Cs-137 ({:.1f}%)'.format(pct_cs137)]
colors_pie = ['#d62728', '#1f77b4']
explode = (0.05, 0.0)
ax.pie(sizes, explode=explode, labels=labels_pie, colors=colors_pie,
       autopct='%1.1f%%', startangle=90, shadow=True,
       textprops={'fontsize': 11})
ax.set_title('(b) Dose contribution fraction')

fig.tight_layout()
fig.savefig('fig_co60_comparison.png',
            bbox_inches='tight', dpi=120)
print('Saved fig_co60_comparison.png')
plt.close(fig)


Saved fig_co60_comparison.png


## 8. Building Shadow Effect Analysis

To demonstrate the building shielding effect, we compute the forward
dose-rate profile along a horizontal transect (y = 200 m) through the
site. The processing facility (Building A) sits at x = 120 m, y = 200 m
and should cast a visible “shadow” behind it (increasing x).

The forward model computes ADER from the SAD using:
- The Fredholm matrix **with** buildings (visibility reduced)
- The Fredholm matrix **without** buildings (full visibility)


In [22]:
# Use the reconstruction results to compute forward dose along a transect
TRANSECT_Y_IDX = NY_FR // 2  # middle row of the Fredholm grid

# Forward dose from SAD with buildings
ader_fwd_bldg = recon_bldg.forward(result_bldg.sad)
# Forward dose from SAD without buildings
ader_fwd_free = recon_free.forward(result_free.sad)

# Extract transect (row TRANSECT_Y_IDX)
transect_bldg = ader_fwd_bldg[TRANSECT_Y_IDX, :]
transect_free = ader_fwd_free[TRANSECT_Y_IDX, :]
transect_x = np.linspace(cell_size/2, SITE_SIZE - cell_size/2, NX_FR)
transect_y = yfr[TRANSECT_Y_IDX]

# Shadow reduction ratio
shadow_ratio = transect_bldg / np.maximum(transect_free, 1e-10)

print('Transect y = {:.1f} m (row {})'.format(transect_y, TRANSECT_Y_IDX))
print('  Building A centre: x = 120 m')
print('  Dose with bldg, at x=120 m : {:.4f} uSv/h'.format(
      transect_bldg[NX_FR // 4]))
print('  Dose w/o bldg,  at x=120 m : {:.4f} uSv/h'.format(
      transect_free[NX_FR // 4]))

# Find minimum shadow ratio behind Building A
behind_a = transect_x > 160  # east of Building A (x=120, width=80)
if behind_a.sum() > 0:
    min_idx = np.where(behind_a)[0][0]
    print('  Min shadow ratio behind Building A: {:.3f} at x={:.0f} m'.format(
        shadow_ratio[behind_a].min(), transect_x[min_idx]))


Transect y = 258.3 m (row 15)
  Building A centre: x = 120 m
  Dose with bldg, at x=120 m : 0.0000 uSv/h
  Dose w/o bldg,  at x=120 m : 0.0000 uSv/h
  Min shadow ratio behind Building A: 0.000 at x=175 m


In [23]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9),
                                gridspec_kw={'height_ratios': [3, 1]})
fig.suptitle('Building Shadow Effect along Transect (y = {:.0f} m)'.format(transect_y),
             fontsize=14, fontweight='bold')

# (a) ADER profile with and without buildings
ax1.plot(transect_x, transect_free * 1000, 'b-', lw=2, alpha=0.7,
         label='Without buildings')
ax1.plot(transect_x, transect_bldg * 1000, 'r-', lw=2.5,
         label='With buildings')
ax1.fill_between(transect_x, transect_bldg * 1000, transect_free * 1000,
                    alpha=0.15, color='blue', label='Shielded region')

# Mark Building A footprint
bldg_a = buildings[0]
ax1.axvspan(bldg_a['x'] - bldg_a['width']/2,
            bldg_a['x'] + bldg_a['width']/2,
            alpha=0.2, color='red', label='Building A')

ax1.set_xlabel('x (m)')
ax1.set_ylabel('ADER [nSv/h]')
ax1.set_title('(a) Forward dose-rate profile with/without buildings')
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, SITE_SIZE)

# (b) Shadow ratio
ax2.axhline(1.0, color='gray', ls='--', lw=1, alpha=0.5)
ax2.plot(transect_x, shadow_ratio, 'k-', lw=1.5)
ax2.fill_between(transect_x, shadow_ratio, 1.0, where=(shadow_ratio < 1.0),
                    alpha=0.3, color='red', label='Shadow region')
ax2.axvspan(bldg_a['x'] - bldg_a['width']/2,
            bldg_a['x'] + bldg_a['width']/2,
            alpha=0.2, color='red')
ax2.set_xlabel('x (m)')
ax2.set_ylabel('Shadow ratio')
ax2.set_title('(b) Shadow ratio (with/without buildings)')
ax2.legend(loc='lower right', fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, SITE_SIZE)
ax2.set_ylim(0, 1.3)

fig.tight_layout()
fig.savefig('fig_co60_shadow.png',
            bbox_inches='tight', dpi=120)
print('Saved fig_co60_shadow.png')
plt.close(fig)


Saved fig_co60_shadow.png


## 9. Summary & Conclusions

1. **Co-60 contamination structure**: The modelled industrial site shows
   a characteristic multi-source contamination pattern with a primary
   hot spot near the processing facility (up to ≈2000 Bq/m²), a
   secondary spot near waste storage, and an elongated pipeline leak.

2. **High kerma constant**: Co-60's Kγ = 137.0 aGy·m²/(s·Bq)
   is 6.4× that of Cs-137 (21.3), meaning even modest surface
   activities produce significant dose rates. The SAKR is
   ≈11.7 nSv/h per kBq/m².

3. **Building shadow effect**: The Fredholm reconstruction with barrier
   geometry shows that buildings redistribute the reconstructed SAD.
   Behind barriers, the forward dose rate is reduced (shadow effect),
   and the inverse problem compensates by redistributing activity.
   The total reconstructed activity differs between the two approaches,
   reflecting the importance of including barrier geometry in real
   decommissioning surveys.

4. **Co-60 dominates dose**: Even when Cs-137 is present at 10% of the
   Co-60 activity, Co-60 accounts for ≈87% of the total gamma dose
   rate due to its much larger kerma constant. This underscores the
   importance of Co-60-specific analysis in activation-product
   contamination scenarios.

5. **Practical implications**: For industrial sites with Co-60
   contamination, the short half-life (5.27 y) means that activity
   decreases by ~50% every 5.3 years. However, high initial activities
   and the large kerma constant mean that dose rates can remain
   significant for several decades. Building geometry must be included
   in radiation surveys for accurate dose assessment.


In [24]:
print('=' * 70)
print('  Co-60 INDUSTRIAL CONTAMINATION — RECONSTRUCTION SUMMARY')
print('=' * 70)
print()
print('{:<50s} {}'.format('Site area',
      '{:.0f} x {:.0f} m = {:.0f} m²'.format(
          SITE_SIZE, SITE_SIZE, SITE_SIZE ** 2)))
print('{:<50s} {}'.format('Measurement points', N_POINTS))
print('{:<50s} {}'.format('Buildings', len(buildings)))
print('{:<50s} {:.1f} Bq/m²'.format('Co-60 median activity', np.median(co60)))
print('{:<50s} {:.1f} Bq/m²'.format('Co-60 max activity', np.max(co60)))
print('{:<50s} {:.4f} uSv/h'.format('Co-60 median dose rate', np.median(dose_co60)))
print('{:<50s} {:.4f} uSv/h'.format('Co-60 max dose rate', np.max(dose_co60)))
print('{:<50s} {:.3f} uSv/h per kBq/m²'.format(
    'SAKR (Co-60)', SAKR_CO60_kBq))
print('{:<50s} {:.1f}%'.format('Co-60 dose share', pct_co60))
print('{:<50s} {:.1f}%'.format('Cs-137 dose share', pct_cs137))
print()
print('{:<50s} {}'.format('Reconstruction (w/ buildings)', result_bldg.method))
print('{:<50s} {:.1e}'.format('  Regularisation alpha', result_bldg.alpha))
print('{:<50s} {:.2e}'.format('  Condition number', result_bldg.info['cond_F']))
print('{:<50s} {:.3e} Bq'.format('  Total activity (Fredholm)',
      result_bldg.total_activity))
print('{:<50s} {:.3e} Bq'.format('  Total activity (MCC)',
      result_bldg.total_activity_mcc))
print('{:<50s} {:.4f}'.format('  Gini (SAD)', result_bldg.info['gini_sad']))
print('{:<50s} {:.4f}'.format('  Compactness ratio',
      result_bldg.info['compactness_ratio']))
print()
print('{:<50s} {:+.1f}%'.format('Building effect on total activity', act_pct))
print('=' * 70)


  Co-60 INDUSTRIAL CONTAMINATION — RECONSTRUCTION SUMMARY

Site area                                          500 x 500 m = 250000 m²
Measurement points                                 80
Buildings                                          3
Co-60 median activity                              0.5 Bq/m²
Co-60 max activity                                 1801.9 Bq/m²
Co-60 median dose rate                             0.0000 uSv/h
Co-60 max dose rate                                0.0000 uSv/h
SAKR (Co-60)                                       0.000 uSv/h per kBq/m²
Co-60 dose share                                   96.2%
Cs-137 dose share                                  3.8%

Reconstruction (w/ buildings)                      fredholm_tikhonov
  Regularisation alpha                             1.0e-10
  Condition number                                 1.06e+00
  Total activity (Fredholm)                        1.881e-03 Bq
  Total activity (MCC)                             2.223e+10 Bq
  